In [0]:
%pip install mlflow pydantic openpyxl

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/query-planner-demo")

Agent = WorkspaceClient()

ALLOWED_CONSTRAINTS_TEXT = """
Allowed dimensions and fields from the facilities_scored table:

location:
- state
- city
- pincode
- lat
- lon

facility_identity:
- facility_type
- operator_type

clinical_specialty:
- specialties

clinical_procedure:
- procedures

clinical_equipment:
- equipment

clinical_capability:
- capabilities

staffing:
- number_doctors
- raw_notes
- reasoning

availability:
- capabilities
- raw_notes

trust:
- trust_score
- trust_flags
- ai_reviewed

evidence:
- raw_notes
- reasoning
"""


PLANNER_PROMPT = f"""
You are a healthcare query planning agent for Indian medical facility search.

You convert a user request into structured search constraints for a downstream retrieval and ranking system.
The downstream system searches a table called facilities_scored.
You must only use dimensions and fields that exist in that table.

Do not recommend any facility.
Do not invent facts.
Do not add fields that are not listed below.
Do not output markdown.
Respond with JSON only.

{ALLOWED_CONSTRAINTS_TEXT}

Return valid JSON with exactly these keys:
medical_need
constraints
urgency
distance_preference
trust_threshold
reasoning_steps

Rules:
- constraints must be an array of objects
- each constraint object must contain exactly these keys:
  - dimension
  - field
  - operator
  - value
  - priority
- dimension must be one of:
  - location
  - facility_identity
  - clinical_specialty
  - clinical_procedure
  - clinical_equipment
  - clinical_capability
  - staffing
  - availability
  - trust
  - evidence
- field must be one of the allowed fields for its dimension
- operator must be one of:
  - equals
  - contains
  - near
  - greater_or_equal
  - less_or_equal
- priority must be one of:
  - hard
  - soft
  - exclude

Interpretation rules:

- Use hard only when the user clearly requires something.
- Use soft when the user expresses preference, desirability, tendency, or flexibility.
- Use exclude when the user explicitly rejects something.
- Location constraints must also appear in constraints.
- If the user asks for "nearest", "closest", or distance-sensitive matching, reflect that in distance_preference.
- If the user mentions a state, city, or pincode explicitly and as a requirement, treat it as hard unless the wording is clearly flexible.
- If the user says "must", "only", "required", "has to", "needs to", "without fail", use hard.
- If the user says "prefer", "preferably", "ideally", "should", "better if possible", use soft.
- If the user says "exclude", "avoid", "not", "must not", "should not", use exclude.

Field-selection guidance:
- Use location for state, city, pincode, coordinates, or geographic intent.
- Use clinical_procedure for specific procedures such as appendectomy, c-section, dialysis, hysterectomy.
- Use clinical_specialty for medical specialties such as anesthesiology, orthopedics, pediatrics, emergency medicine.
- Use clinical_equipment for equipment such as ventilator, CT scanner, oxygen, ICU monitor.
- Use clinical_capability for broader capabilities such as trauma care, emergency surgery, ICU support, NICU support.
- Use availability when the request is about 24/7, always open, round-the-clock, emergency availability, or similar language. Prefer field=raw_notes unless the request clearly maps to capabilities.
- Use staffing when the request refers to doctor count, part-time doctors, visiting doctors, consultants, or staffing model. Prefer field=raw_notes or field=reasoning for staffing model language.
- Use trust when the request refers to confidence, verified quality, suspiciousness, or minimum trust score. For trust_score use numeric operators such as greater_or_equal or less_or_equal.
- Use evidence only when the user explicitly asks for supporting evidence, exact text support, proof, or citations.

Semantic normalization guidance:
- Treat semantically similar phrases as the same intent when planning constraints.
- Examples:
  - "24/7", "24x7", "247", "open 24 hours", "always open", "round the clock" all indicate continuous availability.
  - "appendectomy" and "appendicectomy" refer to the same procedure intent.
  - "part-time doctors", "visiting doctors", and "visiting consultants" may indicate a similar staffing pattern.
- Prefer the canonical user intent in the value field, not every variant.

Output requirements:
- medical_need should be a concise phrase describing the core medical need.
- trust_threshold must be a number between 0 and 1. If the user gives no trust preference, choose a sensible default such as 0.6.
- urgency must be one of: low, medium, high, unknown.
- distance_preference must be an object with exactly these keys:
  - nearest_only
  - max_distance_km
- reasoning_steps must briefly explain why major constraints were classified as hard, soft, or exclude.
- If a value is unknown, use null.
- If there are no constraints for a category, do not invent any.
- For queries involving "rural", use evidence/raw_notes unless a structured rural field exists.
- Do not duplicate equivalent constraints unless they target different fields for a valid reason.

- reasoning_steps must be an array of strings
- each item should explain one important planning decision
- reasoning_steps must briefly explain why major constraints were classified as hard, soft, or exclude

Examples:

User: "Find a hospital in Bihar that must be open 24/7 and can do appendectomy"
Output constraint examples:
{{"dimension":"location","field":"state","operator":"equals","value":"Bihar","priority":"hard"}}
{{"dimension":"availability","field":"raw_notes","operator":"contains","value":"24/7","priority":"hard"}}
{{"dimension":"clinical_procedure","field":"procedures","operator":"contains","value":"appendectomy","priority":"hard"}}

User: "Preferably in Patna and ideally open 24/7"
Output constraint examples:
{{"dimension":"location","field":"city","operator":"equals","value":"Patna","priority":"soft"}}
{{"dimension":"availability","field":"raw_notes","operator":"contains","value":"24/7","priority":"soft"}}

User: "Avoid ayurvedic facilities"
Output constraint example:
{{"dimension":"facility_identity","field":"facility_type","operator":"contains","value":"ayurvedic","priority":"exclude"}}
"""


@mlflow.trace
def plan_query(user_query: str):
    response = Agent.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=PLANNER_PROMPT),
            ChatMessage(role=ChatMessageRole.USER, content=user_query),
        ],
        temperature=0.1,
        max_tokens=900
    )
    return response.choices[0].message.content

user_query = "Find the nearest facility in rural Bihar that can perform an emergency appendectomy and typically leverages parttime doctors. It preferably should be open 24/7"
raw_plan = plan_query(user_query)
print(raw_plan)

In [0]:
import json
import re
from typing import Dict, List, Any, Tuple

import mlflow
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

In [0]:
@mlflow.trace
def parse_json_safely(raw_text: str) -> Dict[str, Any]:
    if raw_text is None:
        raise ValueError("Planner output is None")

    text = raw_text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError("No valid JSON object found in planner output")

In [0]:
ALLOWED_DIMENSIONS = {
    "location",
    "facility_identity",
    "clinical_specialty",
    "clinical_procedure",
    "clinical_equipment",
    "clinical_capability",
    "staffing",
    "availability",
    "trust",
    "evidence",
}

ALLOWED_OPERATORS = {
    "equals",
    "contains",
    "near",
    "greater_or_equal",
    "less_or_equal",
}

ALLOWED_PRIORITIES = {
    "hard",
    "soft",
    "exclude",
}

@mlflow.trace
def validate_query_plan(query_plan: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(query_plan, dict):
        raise ValueError("query_plan must be a dict")

    if "constraints" not in query_plan or not isinstance(query_plan["constraints"], list):
        raise ValueError("query_plan must contain a constraints list")

    if "reasoning_steps" in query_plan and not isinstance(query_plan["reasoning_steps"], list):
        raise ValueError("reasoning_steps must be an array")

    for c in query_plan["constraints"]:
        if c.get("dimension") not in ALLOWED_DIMENSIONS:
            raise ValueError(f"Invalid dimension: {c.get('dimension')}")
        if c.get("operator") not in ALLOWED_OPERATORS:
            raise ValueError(f"Invalid operator: {c.get('operator')}")
        if c.get("priority") not in ALLOWED_PRIORITIES:
            raise ValueError(f"Invalid priority: {c.get('priority')}")

    return query_plan

In [0]:
@mlflow.trace
def split_constraints(query_plan: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    constraints = query_plan.get("constraints", []) or []

    hard_constraints = [c for c in constraints if c.get("priority") == "hard"]
    soft_constraints = [c for c in constraints if c.get("priority") == "soft"]
    exclude_constraints = [c for c in constraints if c.get("priority") == "exclude"]

    return hard_constraints, soft_constraints, exclude_constraints

In [0]:
@mlflow.trace
def build_search_df(table_name: str = "facilities_scored") -> DataFrame:
    df = spark.table(table_name)

    return (
        df
        .withColumn("state_text", F.lower(F.coalesce(F.col("state"), F.lit(""))))
        .withColumn("city_text", F.lower(F.coalesce(F.col("city"), F.lit(""))))
        .withColumn("pincode_text", F.coalesce(F.col("pincode"), F.lit("")))
        .withColumn("facility_type_text", F.lower(F.coalesce(F.col("facility_type"), F.lit(""))))
        .withColumn("operator_type_text", F.lower(F.coalesce(F.col("operator_type"), F.lit(""))))
        .withColumn("specialties_text", F.lower(F.concat_ws(" ", F.col("specialties"))))
        .withColumn("procedures_text", F.lower(F.concat_ws(" ", F.col("procedures"))))
        .withColumn("equipment_text", F.lower(F.concat_ws(" ", F.col("equipment"))))
        .withColumn("capabilities_text", F.lower(F.concat_ws(" ", F.col("capabilities"))))
        .withColumn("trust_flags_text", F.lower(F.concat_ws(" || ", F.col("trust_flags"))))
        .withColumn("raw_notes_text", F.lower(F.coalesce(F.col("raw_notes"), F.lit(""))))
        .withColumn("reasoning_text", F.lower(F.coalesce(F.col("reasoning"), F.lit(""))))
        .withColumn(
            "search_text",
            F.concat_ws(
                " ",
                F.col("specialties_text"),
                F.col("procedures_text"),
                F.col("equipment_text"),
                F.col("capabilities_text"),
                F.col("raw_notes_text"),
                F.col("reasoning_text"),
                F.col("trust_flags_text"),
            )
        )
        .withColumn("trust_score_safe", F.coalesce(F.col("trust_score"), F.lit(0.0)))
        .withColumn("ai_reviewed_safe", F.coalesce(F.col("ai_reviewed"), F.lit(False)))
    )

In [0]:
SEMANTIC_SYNONYMS = {
    "24/7": ["24/7", "24x7", "247", "24 hours", "24 hour", "open 24 hours", "always open", "round the clock"],
    "appendectomy": ["appendectomy", "appendicectomy", "appendix removal", "appendix removal surgery", "appendicitis surgery"],
    "parttime doctors": ["part-time doctors", "part time doctors", "parttime doctors", "visiting doctor", "visiting doctors", "visiting consultant", "visiting consultants"],
    "icu": ["icu", "intensive care", "critical care", "nicu", "picu"],
}

def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip().lower()

def expand_semantic_values(value: Any) -> List[str]:
    base = normalize_text(value)
    if not base:
        return []
    return list(dict.fromkeys([base] + SEMANTIC_SYNONYMS.get(base, [])))

In [0]:
def _broad_expr(field: str, operator: str, value: Any):
    v = normalize_text(value)

    field_map = {
        "state": F.col("state_text"),
        "city": F.col("city_text"),
        "pincode": F.col("pincode_text"),
        "facility_type": F.col("facility_type_text"),
        "operator_type": F.col("operator_type_text"),
        "specialties": F.col("specialties_text"),
        "procedures": F.col("procedures_text"),
        "equipment": F.col("equipment_text"),
        "capabilities": F.col("capabilities_text"),
        "raw_notes": F.col("raw_notes_text"),
        "reasoning": F.col("reasoning_text"),
        "trust_flags": F.col("trust_flags_text"),
        "trust_score": F.col("trust_score_safe"),
    }

    col_expr = field_map[field]

    if operator == "equals":
        return col_expr == v

    if operator == "contains":
        variants = expand_semantic_values(v)
        expr = None
        for term in variants:
            current = col_expr.contains(term)
            expr = current if expr is None else (expr | current)
        return expr

    if operator == "greater_or_equal":
        return col_expr >= float(value)

    if operator == "less_or_equal":
        return col_expr <= float(value)

    return None

In [0]:
@mlflow.trace
def retrieve_candidate_pool(query_plan: Dict[str, Any], table_name: str = "facilities_scored", max_candidates: int = 40):
    hard_constraints, soft_constraints, exclude_constraints = split_constraints(query_plan)

    df = build_search_df(table_name)

    for c in hard_constraints:
        if c.get("operator") == "near":
            continue

        if c["field"] in {"state", "city", "pincode", "trust_score", "facility_type", "operator_type"}:
            expr = _broad_expr(c["field"], c["operator"], c["value"])
            if expr is not None:
                df = df.filter(expr)

    for c in exclude_constraints:
        if c.get("operator") == "near":
            continue

        expr = _broad_expr(c["field"], c["operator"], c["value"])
        if expr is not None:
            df = df.filter(~expr)

    text_terms = []
    for c in hard_constraints + soft_constraints:
        if c["field"] in {"specialties", "procedures", "equipment", "capabilities", "raw_notes", "reasoning"}:
            text_terms.extend(expand_semantic_values(c["value"]))

    if text_terms:
        expr = None
        for term in set(text_terms):
            current = F.col("search_text").contains(term)
            expr = current if expr is None else (expr | current)
        df = df.filter(expr)

    rows = [r.asDict(recursive=True) for r in df.limit(max_candidates).collect()]
    return rows

In [0]:
Agent = WorkspaceClient()

CANDIDATE_JUDGE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "candidate_judgement",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "match_score": {"type": "number"},
                "trust_penalty": {"type": "number"},
                "should_recommend": {"type": "boolean"},
                "matched_requirements": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "conflicting_flags": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "reasoning": {"type": "string"}
            },
            "required": [
                "match_score",
                "trust_penalty",
                "should_recommend",
                "matched_requirements",
                "conflicting_flags",
                "reasoning"
            ],
            "additionalProperties": False
        }
    }
}

@mlflow.trace
def llm_judge_candidate(user_query: str, query_plan: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    facility_payload = {
        "name": candidate.get("name"),
        "state": candidate.get("state"),
        "city": candidate.get("city"),
        "facility_type": candidate.get("facility_type"),
        "operator_type": candidate.get("operator_type"),
        "specialties": candidate.get("specialties"),
        "procedures": candidate.get("procedures"),
        "equipment": candidate.get("equipment"),
        "capabilities": candidate.get("capabilities"),
        "raw_notes": candidate.get("raw_notes"),
        "trust_score": candidate.get("trust_score"),
        "trust_flags": candidate.get("trust_flags"),
        "reasoning": candidate.get("reasoning"),
        "ai_reviewed": candidate.get("ai_reviewed"),
    }

    system_prompt = """
You are a medical facility relevance judge.

Your job is to compare ONE facility against ONE user healthcare query and structured query plan.

Be careful with semantic equivalence:
- "24/7", "24x7", "247", "always open", and "open 24 hours" may indicate similar availability.
- "appendectomy" and "appendicectomy" may indicate the same procedure intent.
- "part-time doctors", "visiting doctors", and "visiting consultants" may indicate similar staffing patterns.

Important:
- trust_flags matter only when they conflict with the user's actual needs.
- Do not reject a facility only because it has flags; reject it when the flags undermine the requested medical need.
- Prefer evidence from procedures, capabilities, raw_notes, and reasoning.
- Be conservative: if support is weak, reduce match_score.

Scoring:
- match_score: 0.0 to 1.0
- trust_penalty: 0.0 to 1.0
"""

    user_payload = {
        "user_query": user_query,
        "query_plan": query_plan,
        "candidate": facility_payload,
    }

    response = Agent.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
            ChatMessage(role=ChatMessageRole.USER, content=json.dumps(user_payload, ensure_ascii=False)),
        ],
        response_format=CANDIDATE_JUDGE_SCHEMA,
        temperature=0.1,
        max_tokens=500
    )

    content = response.choices[0].message.content
    if isinstance(content, str):
        return json.loads(content)
    return content

In [0]:
@mlflow.trace
def rerank_candidates_with_llm(user_query: str, query_plan: Dict[str, Any], candidates: List[Dict[str, Any]], top_k: int = 10):
    judged = []

    for candidate in candidates[:top_k]:
        judgement = llm_judge_candidate(user_query, query_plan, candidate)

        base_trust = float(candidate.get("trust_score") or 0.0)
        match_score = float(judgement["match_score"])
        trust_penalty = float(judgement["trust_penalty"])

        final_score = (0.55 * match_score) + (0.30 * base_trust) - (0.25 * trust_penalty)
        final_score = max(0.0, min(1.0, round(final_score, 4)))

        judged.append({
            "name": candidate.get("name"),
            "state": candidate.get("state"),
            "city": candidate.get("city"),
            "facility_type": candidate.get("facility_type"),
            "final_score": final_score,
            "base_trust_score": base_trust,
            "judge": judgement,
            "trust_flags": candidate.get("trust_flags"),
            "raw_notes": candidate.get("raw_notes"),
        })

    judged.sort(key=lambda x: x["final_score"], reverse=True)
    return judged

In [0]:
user_query = "Find the nearest facility in rural Bihar that can perform an emergency appendectomy and typically leverages parttime doctors. It preferably should be open 24/7"

raw_plan = plan_query(user_query)
query_plan = parse_json_safely(raw_plan)
query_plan = validate_query_plan(query_plan)

candidate_pool = retrieve_candidate_pool(query_plan, table_name="facilities_scored", max_candidates=40)
ranked_results = rerank_candidates_with_llm(user_query, query_plan, candidate_pool, top_k=12)

ranked_results[:3]